In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path 
import sys,importlib
import re,ast
import joblib,openpyxl,inspect
import json , shutil
import importlib.metadata as metadata

In [4]:
ROOT = Path("/workspace")
excel = ROOT / "data/raw/02_CortexHR_AI_ML_Dataset.xlsx"

In [5]:
scoring_df = pd.read_excel(excel,sheet_name="attrition_scoring_2026")

In [6]:
sample_df = pd.read_excel(excel,sheet_name="sample_attrition_risk")

In [55]:
scoring_df.shape

(430, 34)

In [8]:
 sample_df.shape

(430, 11)

In [9]:
metadata_path = ROOT / "models/attrition-v2_metadata.json"

In [11]:
model_path = ROOT / "models/attrition-v2.joblib"

In [12]:
with open(metadata_path) as f:metadata = json.load(f)

In [13]:
attrition_model = joblib.load(model_path)

In [14]:
expected_features = (metadata["numeric_features"] + metadata["categorical_features"])

In [16]:
missing_features = [
    col for col in expected_features
    if col not in scoring_df.columns
]
extra_columns = [
    col for col in scoring_df.columns
    if col not in expected_features
]


In [17]:
metadata["model_version"]

'attrition-v2'

In [18]:
metadata["model_type"]

'XGBoost with sigmoid calibration'

In [22]:
for i in missing_features:
    print("-", i)


- attendance_scheduled_days
- attendance_present_days
- attendance_wfh_days
- attendance_half_days_unpaid
- attendance_paid_leave_days
- attendance_unpaid_absence_days
- attendance_longest_unpaid_streak_days
- attendance_late_arrivals
- attendance_avg_hours_worked
- attendance_overtime_hours
- attendance_attendance_rate
- payroll_monthly_fixed_gross
- payroll_payable_days
- payroll_lop_days
- payroll_overtime_hours_paid
- payroll_overtime_pay
- payroll_variable_pay
- payroll_gross_pay
- payroll_pf_employee
- payroll_esi_employee
- payroll_professional_tax
- payroll_tds
- payroll_other_deductions
- payroll_total_deductions
- payroll_net_pay
- tenure_years
- raise_overdue
- attendance_gap
- unpaid_per_month
- overtime_load


In [23]:
extra_columns

['row_id', 'employee_id', 'observation_date']

In [24]:
 type(attrition_model).__name__

'CalibratedClassifierCV'

In [49]:
# Task 14: Load forecasting source data

headcount_df = pd.read_excel(
    excel,
    sheet_name="monthly_headcount_history"
)

workforce_events_df = pd.read_excel(
    excel,
    sheet_name="workforce_events"
)

sample_forecast_df = pd.read_excel(
    excel,
    sheet_name="sample_forecast"
)

for name, df in {
    "Monthly Headcount History": headcount_df,
    "Workforce Events": workforce_events_df,
    "Sample Forecast": sample_forecast_df
}.items():
    print(f"\n{'='*55}")
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())



Monthly Headcount History
Shape: (972, 14)
Columns: ['month', 'department_id', 'department_name', 'headcount_end', 'hires', 'exits', 'voluntary_exits', 'transfers_in', 'transfers_out', 'headcount_start', 'hiring_drive_flag', 'attrition_rate_monthly_pct', 'appraisal_cycle_flag', 'financial_year_end_flag']

First 5 rows:


,month,department_id,department_name,headcount_end,hires,exits,voluntary_exits,transfers_in,transfers_out,headcount_start,hiring_drive_flag,attrition_rate_monthly_pct,appraisal_cycle_flag,financial_year_end_flag
0,2020-01-01,ALL,Company total,171,5,4,4,0,0,170,0,2.35,0,0
1,2020-01-01,D00,Executive Office,1,0,0,0,0,0,1,0,0.00,0,0
2,2020-01-01,D01,Engineering,60,1,2,2,0,0,61,0,3.28,0,0
3,2020-01-01,D02,Product & Design,8,0,0,0,0,0,8,0,0.00,0,0
4,2020-01-01,D03,Data & AI,16,0,0,0,0,0,16,0,0.00,0,0



Workforce Events
Shape: (9, 6)
Columns: ['event_id', 'start_date', 'end_date', 'department_id', 'event_name', 'description']

First 5 rows:


,event_id,start_date,end_date,department_id,event_name,description
0,EV-01,2020-04-01,2020-06-30,ALL,COVID-19 hiring freeze,Hiring almost stopped; exits slowed
1,EV-02,2021-04-01,2021-04-30,ALL,Reduced increment cycle,Average increments cut by about 25%
2,EV-03,2022-09-01,2022-11-30,D06,Support scale-up for festive season,Large batch of Support Associates hired
3,EV-04,2023-07-01,2023-07-31,ALL,Hybrid work policy introduced,Three-day office policy for most roles
4,EV-05,2023-07-01,2023-08-31,D01,Engineering hiring drive,Hiring doubled for two months



Sample Forecast
Shape: (72, 10)
Columns: ['forecast_id', 'metric', 'department_id', 'forecast_month', 'forecast_value', 'lower_80', 'upper_80', 'method', 'trained_through', 'model_version']

First 5 rows:


,forecast_id,metric,department_id,forecast_month,forecast_value,lower_80,upper_80,method,trained_through,model_version
0,FC-0001,headcount_end,ALL,2026-10-01,426.3,417.5,435.1,linear trend (36m) + half-strength seasonality,2026-09,trend-seasonal-baseline-0.1
1,FC-0002,headcount_end,ALL,2026-11-01,429.7,419.9,439.5,linear trend (36m) + half-strength seasonality,2026-09,trend-seasonal-baseline-0.1
2,FC-0003,headcount_end,ALL,2026-12-01,431.4,420.7,442.2,linear trend (36m) + half-strength seasonality,2026-09,trend-seasonal-baseline-0.1
3,FC-0004,headcount_end,ALL,2027-01-01,436.0,424.4,447.6,linear trend (36m) + half-strength seasonality,2026-09,trend-seasonal-baseline-0.1
4,FC-0005,headcount_end,ALL,2027-02-01,439.7,427.3,452.1,linear trend (36m) + half-strength seasonality,2026-09,trend-seasonal-baseline-0.1


In [51]:
import os
import glob
import pandas as pd

print("MODEL FILES")
for file in glob.glob("/workspace/models/**/*", recursive=True):
    if os.path.isfile(file):
        print(file)

print("\nTASK 14 SOURCE DATA")
for sheet in ["attrition_scoring_2026", "sample_attrition_risk"]:
    df_check = pd.read_excel(
        ,
        sheet_name=sheet
    )
    print(f"\n{sheet}: {df_check.shape}")
    print(df_check.columns.tolist())
    display(df_check.head(3))

MODEL FILES
/workspace/models/attendance_detector.joblib
/workspace/models/attrition-v2.joblib
/workspace/models/attrition-v2_metadata.json
/workspace/models/attrition_preprocessor.joblib
/workspace/models/chatbot_bundle.joblib
/workspace/models/metadata.json
/workspace/models/payroll_detector.joblib
/workspace/models/reason_codes.joblib
/workspace/models/resume_ranking_bundle.joblib

TASK 14 SOURCE DATA

attrition_scoring_2026: (430, 34)
['row_id', 'employee_id', 'observation_date', 'department_id', 'role_id', 'job_level', 'work_location', 'work_mode', 'employment_type', 'highest_education', 'tenure_months', 'total_experience_years', 'previous_employers_count', 'commute_distance_km', 'annual_fixed_pay', 'comp_ratio', 'raises_received', 'last_raise_pct', 'months_since_last_raise', 'promotions_received', 'months_since_last_promotion', 'internal_transfers', 'last_performance_rating', 'last_engagement_score', 'manager_changes_12m', 'training_hours_12m', 'attendance_rate_3m', 'late_arrival

,row_id,employee_id,observation_date,department_id,role_id,job_level,work_location,work_mode,employment_type,highest_education,...,manager_changes_12m,training_hours_12m,attendance_rate_3m,late_arrivals_3m,avg_overtime_hours_per_month_3m,avg_hours_worked_3m,wfh_share_3m,unpaid_leave_days_6m,paid_leave_days_6m,dept_attrition_rate_prior_12m
0,1,EMP-0002,2026-09-30,D08,R047,4,Pune,Remote,Permanent,B.E. Computer Science,...,0,34.0,0.9615,4,0.0,8.36,0.908,2.5,1,0.1765
1,2,EMP-0004,2026-09-30,D03,R017,3,Erode,Hybrid,Permanent,MCA,...,0,48.1,0.8923,1,0.0,7.83,0.385,0.0,10,0.1346
2,3,EMP-0011,2026-09-30,D07,R039,1,Kochi,Hybrid,Permanent,M.A. Social Work,...,0,57.7,0.8923,3,3.0,7.78,0.446,2.0,5,0.1500



sample_attrition_risk: (430, 11)
['prediction_id', 'employee_id', 'department_id', 'as_of_date', 'risk_score_12m', 'risk_band', 'top_reason_1', 'top_reason_2', 'top_reason_3', 'model_version', 'model_note']


,prediction_id,employee_id,department_id,as_of_date,risk_score_12m,risk_band,top_reason_1,top_reason_2,top_reason_3,model_version,model_note
0,AR-0001,EMP-0002,D08,2026-09-30,0.035,Low,Total experience years (31.8),Low engagement score (2.2/5),Wfh share 3m (0.9),attrition-logreg-baseline-0.1,"Baseline demo model: a support signal for HR, ..."
1,AR-0002,EMP-0004,D03,2026-09-30,0.074,Low,Low engagement score (1.0/5),Total experience years (18.5),Pay is 0.90x the role midpoint,attrition-logreg-baseline-0.1,"Baseline demo model: a support signal for HR, ..."
2,AR-0003,EMP-0011,D07,2026-09-30,0.035,Low,Low engagement score (2.2/5),Long commute (44 km),Total experience years (13.8),attrition-logreg-baseline-0.1,"Baseline demo model: a support signal for HR, ..."


In [52]:
scoring_check = pd.read_excel(excel,sheet_name="attrition_scoring_2026")
sample_check = pd.read_excel(excel,sheet_name="sample_attrition_risk")
expected_output_columns = [
    "prediction_id",
    "employee_id",
    "department_id",
    "as_of_date",
    "risk_score_12m",
    "risk_band",
    "top_reason_1",
    "top_reason_2",
    "top_reason_3",
    "model_version",
    "model_note"
]
print("SCORING INPUT")
print("Rows:", len(scoring_check))
print("Columns:", len(scoring_check.columns))
print("\nREFERENCE OUTPUT")
print("Rows:", len(sample_check))
print("Columns:", len(sample_check.columns))
print("Exact schema:", sample_check.columns.tolist() == expected_output_columns)
print("\nEMPLOYEE MATCH:")
print(
    set(scoring_check["employee_id"]) ==
    set(sample_check["employee_id"])
)
print("\nMISSING VALUES:")
print(sample_check[expected_output_columns].isna().sum())

SCORING INPUT
Rows: 430
Columns: 34

REFERENCE OUTPUT
Rows: 430
Columns: 11
Exact schema: True

EMPLOYEE MATCH:
True

MISSING VALUES:
prediction_id     0
employee_id       0
department_id     0
as_of_date        0
risk_score_12m    0
risk_band         0
top_reason_1      0
top_reason_2      0
top_reason_3      0
model_version     0
model_note        0
dtype: int64


In [53]:
with open( "/workspace/models/attrition-v2_metadata.json","r") as f: attrition_v2_metadata = json.load(f)
print("========== V2 MODEL METADATA ==========")
print(json.dumps(attrition_v2_metadata, indent=2, default=str))
# Load saved artifacts
attrition_v2_model = joblib.load(
    "/workspace/models/attrition-v2.joblib"
)
attrition_preprocessor = joblib.load(
    "/workspace/models/attrition_preprocessor.joblib"
)

print("\n========== MODEL STRUCTURE ==========")
print("Model type:", type(attrition_v2_model))

if isinstance(attrition_v2_model, dict):
    print("Model bundle keys:", list(attrition_v2_model.keys()))

print("Model input feature count:",
      getattr(attrition_v2_model, "n_features_in_", "Not available"))

print("Model input feature names:",
      getattr(attrition_v2_model, "feature_names_in_", "Not available"))

print("Model classes:",
      getattr(attrition_v2_model, "classes_", "Not available"))

print("\n========== PREPROCESSOR STRUCTURE ==========")
print("Preprocessor type:", type(attrition_preprocessor))

print("Preprocessor input feature count:",
      getattr(attrition_preprocessor, "n_features_in_", "Not available"))

print("Preprocessor input feature names:",
      getattr(attrition_preprocessor, "feature_names_in_", "Not available"))

if hasattr(attrition_preprocessor, "transformers_"):
    for name, transformer, columns in attrition_preprocessor.transformers_:
        print(f"{name}: {len(columns)} input columns")
        print(list(columns))

========== V2 MODEL METADATA ==========
{
  "model_version": "attrition-v2",
  "model_type": "XGBoost with sigmoid calibration",
  "target": "left_within_12m",
  "training_rows": 2389,
  "feature_count": 61,
  "positive_cases": 349,
  "negative_cases": 2040,
  "positive_rate": 0.1460862285475094,
  "numeric_features": [
    "job_level",
    "tenure_months",
    "total_experience_years",
    "previous_employers_count",
    "commute_distance_km",
    "annual_fixed_pay",
    "comp_ratio",
    "raises_received",
    "last_raise_pct",
    "months_since_last_raise",
    "promotions_received",
    "months_since_last_promotion",
    "internal_transfers",
    "last_performance_rating",
    "last_engagement_score",
    "manager_changes_12m",
    "training_hours_12m",
    "attendance_rate_3m",
    "late_arrivals_3m",
    "avg_overtime_hours_per_month_3m",
    "avg_hours_worked_3m",
    "wfh_share_3m",
    "unpaid_leave_days_6m",
    "paid_leave_days_6m",
    "dept_attrition_rate_prior_12m",
    "

In [54]:
# Compare v2 model requirements with the 430 scoring records

v2_expected_features = list(
    attrition_v2_model.feature_names_in_
)

available_features = set(scoring_check.columns)

present_features = [
    col for col in v2_expected_features
    if col in available_features
]

missing_features = [
    col for col in v2_expected_features
    if col not in available_features
]

print("V2 MODEL INPUT CHECK")
print("Expected features:", len(v2_expected_features))
print("Available in scoring data:", len(present_features))
print("Missing features:", len(missing_features))

print("\nMISSING FEATURES:")
for feature in missing_features:
    print("-", feature)

print("\nV2 BASE ESTIMATOR")
print("Estimator:", attrition_v2_model.estimator)
print("Estimator type:", type(attrition_v2_model.estimator))

print("\nCALIBRATED ESTIMATOR DETAILS")
for i, calibrated in enumerate(
    attrition_v2_model.calibrated_classifiers_[:1]
):
    print("Calibrated classifier type:", type(calibrated))
    print("Internal attributes:", calibrated.__dict__.keys())

    fitted_estimator = getattr(calibrated, "estimator", None)
    print("Fitted estimator type:", type(fitted_estimator))
    print("Fitted estimator:", fitted_estimator)

V2 MODEL INPUT CHECK
Expected features: 61
Available in scoring data: 31
Missing features: 30

MISSING FEATURES:
- attendance_scheduled_days
- attendance_present_days
- attendance_wfh_days
- attendance_half_days_unpaid
- attendance_paid_leave_days
- attendance_unpaid_absence_days
- attendance_longest_unpaid_streak_days
- attendance_late_arrivals
- attendance_avg_hours_worked
- attendance_overtime_hours
- attendance_attendance_rate
- payroll_monthly_fixed_gross
- payroll_payable_days
- payroll_lop_days
- payroll_overtime_hours_paid
- payroll_overtime_pay
- payroll_variable_pay
- payroll_gross_pay
- payroll_pf_employee
- payroll_esi_employee
- payroll_professional_tax
- payroll_tds
- payroll_other_deductions
- payroll_total_deductions
- payroll_net_pay
- tenure_years
- raise_overdue
- attendance_gap
- unpaid_per_month
- overtime_load

V2 BASE ESTIMATOR
Estimator: Pipeline(steps=[('preprocessing',
                 Pipeline(steps=[('preprocessor',
                                  ColumnTr

In [39]:
from pathlib import Path
import pandas as pd

# Original source workbook
EXCEL_PATH = Path(
    "/workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx"
)
if not EXCEL_PATH.exists():
    print("Workbook not found:", EXCEL_PATH)
else:
    print("Workbook found:", EXCEL_PATH)

    attrition_raw = pd.read_excel(
        EXCEL_PATH,
        sheet_name="attrition_dataset"
    )

    attendance_raw = pd.read_excel(
        EXCEL_PATH,
        sheet_name="attendance_monthly_features"
    )

    payroll_raw = pd.read_excel(
        EXCEL_PATH,
        sheet_name="payroll_monthly_features"
    )

    datasets = {
        "ATTRITION": attrition_raw,
        "ATTENDANCE": attendance_raw,
        "PAYROLL": payroll_raw
    }

    for name, data in datasets.items():
        print("\n" + "=" * 60)
        print(name)
        print("=" * 60)
        print("Shape:", data.shape)
        print("Columns:", data.columns.tolist())

    employee_id = "EMP-0002"

    for name, data in [
        ("ATTENDANCE", attendance_raw),
        ("PAYROLL", payroll_raw)
    ]:
        print("\n" + "=" * 60)
        print(name, "- EMP-0002 HISTORY")
        print("=" * 60)

        employee_data = data[
            data["employee_id"].astype(str) == employee_id
        ].copy()

        employee_data["month"] = pd.to_datetime(
            employee_data["month"],
            errors="coerce"
        )

        display(
            employee_data.sort_values("month")
        )

Workbook found: /workspace/data/raw/02_CortexHR_AI_ML_Dataset.xlsx

ATTRITION
Shape: (2389, 42)
Columns: ['row_id', 'employee_id', 'observation_date', 'department_id', 'role_id', 'job_level', 'work_location', 'work_mode', 'employment_type', 'highest_education', 'tenure_months', 'total_experience_years', 'previous_employers_count', 'commute_distance_km', 'annual_fixed_pay', 'comp_ratio', 'raises_received', 'last_raise_pct', 'months_since_last_raise', 'promotions_received', 'months_since_last_promotion', 'internal_transfers', 'last_performance_rating', 'last_engagement_score', 'manager_changes_12m', 'training_hours_12m', 'attendance_rate_3m', 'late_arrivals_3m', 'avg_overtime_hours_per_month_3m', 'avg_hours_worked_3m', 'wfh_share_3m', 'unpaid_leave_days_6m', 'paid_leave_days_6m', 'dept_attrition_rate_prior_12m', 'left_within_12m', 'voluntary_left_within_12m', 'source', 'exit_date', 'resignation_date', 'exit_reason', 'exit_interview_done', 'fnf_settlement_amount']

ATTENDANCE
Shape: (7339

,attendance_feature_id,employee_id,month,department_id,job_level,scheduled_days,present_days,wfh_days,half_days_unpaid,paid_leave_days,unpaid_absence_days,longest_unpaid_streak_days,late_arrivals,avg_hours_worked,overtime_hours,attendance_rate
0,AF-00001,EMP-0002,2025-04-01,D08,4,20,20,18,0,0,0,0,1,8.50,0.0,1.0000
1,AF-00002,EMP-0002,2025-05-01,D08,4,21,20,18,1,0,0,0,2,8.71,0.0,0.9762
2,AF-00003,EMP-0002,2025-06-01,D08,4,21,19,18,1,1,0,0,1,8.25,0.0,0.9286
3,AF-00004,EMP-0002,2025-07-01,D08,4,23,23,22,0,0,0,0,1,8.63,0.0,1.0000
4,AF-00005,EMP-0002,2025-08-01,D08,4,19,19,19,0,0,0,0,1,8.62,0.0,1.0000
5,AF-00006,EMP-0002,2025-09-01,D08,4,22,17,16,0,4,1,1,1,6.84,0.0,0.7727
6,AF-00007,EMP-0002,2025-10-01,D08,4,20,19,18,0,0,1,1,1,8.57,0.0,0.9500
7,AF-00008,EMP-0002,2025-11-01,D08,4,20,19,19,0,1,0,0,1,8.20,0.0,0.9500
8,AF-00009,EMP-0002,2025-12-01,D08,4,22,21,20,0,1,0,0,1,8.31,0.0,0.9545
9,AF-00010,EMP-0002,2026-01-01,D08,4,19,19,19,0,0,0,0,1,8.78,0.0,1.0000



PAYROLL - EMP-0002 HISTORY


,payroll_item_id,run_id,employee_id,month,department_id,role_id,job_level,overtime_eligible,monthly_fixed_gross,payable_days,...,variable_pay,variable_pay_type,gross_pay,pf_employee,esi_employee,professional_tax,tds,other_deductions,total_deductions,net_pay
0,PI-000001,PR-2025-04,EMP-0002,2025-04-01,D08,R047,4,N,144460.0,30.0,...,34670.4,Performance Bonus,179130.40,1800.0,0.0,200,18625.79,0,20625.79,158504.61
386,PI-000387,PR-2025-05,EMP-0002,2025-05-01,D08,R047,4,N,144460.0,30.5,...,0.0,NaN,142130.00,1800.0,0.0,200,11230.24,0,13230.24,128899.76
775,PI-000773,PR-2025-06,EMP-0002,2025-06-01,D08,R047,4,N,144460.0,29.5,...,0.0,NaN,142052.33,1800.0,0.0,200,11224.11,0,13224.11,128828.22
1168,PI-001165,PR-2025-07,EMP-0002,2025-07-01,D08,R047,4,N,144460.0,31.0,...,0.0,NaN,144460.00,1800.0,0.0,200,11414.35,0,13414.35,131045.65
1563,PI-001559,PR-2025-08,EMP-0002,2025-08-01,D08,R047,4,N,144460.0,31.0,...,0.0,NaN,144460.00,1800.0,0.0,200,11414.35,0,13414.35,131045.65
1962,PI-001958,PR-2025-09,EMP-0002,2025-09-01,D08,R047,4,N,144460.0,29.0,...,0.0,NaN,139644.67,1800.0,0.0,200,11033.87,0,13033.87,126610.80
2359,PI-002355,PR-2025-10,EMP-0002,2025-10-01,D08,R047,4,N,144460.0,30.0,...,10000.0,Festival Bonus,149800.01,1800.0,0.0,200,13126.14,0,15126.14,134673.87
2763,PI-002758,PR-2025-11,EMP-0002,2025-11-01,D08,R047,4,N,144460.0,30.0,...,0.0,NaN,144460.00,1800.0,0.0,200,11414.35,0,13414.35,131045.65
3162,PI-003157,PR-2025-12,EMP-0002,2025-12-01,D08,R047,4,N,144460.0,31.0,...,0.0,NaN,144460.00,1800.0,0.0,200,11414.35,0,13414.35,131045.65
3565,PI-003560,PR-2026-01,EMP-0002,2026-01-01,D08,R047,4,N,144460.0,31.0,...,0.0,NaN,144460.00,1800.0,0.0,200,11414.35,0,13414.35,131045.65


In [40]:
import pandas as pd
import numpy as np

# Reload saved output and source sheets
saved_df = pd.read_csv(
    "/workspace/data/processed/attrition_features.csv"
)

saved_df["observation_date"] = pd.to_datetime(
    saved_df["observation_date"]
)

attendance_raw["month"] = pd.to_datetime(
    attendance_raw["month"]
)

payroll_raw["month"] = pd.to_datetime(
    payroll_raw["month"]
)

employee_id = "EMP-0002"
observation_date = pd.Timestamp("2025-09-30")

# Exclude the observation month.
observation_month = observation_date.to_period("M").to_timestamp()
window_start = observation_month - pd.DateOffset(months=3)

print("Expected history window:")
print(window_start, "to", observation_month)

# Get saved employee record
saved_row = saved_df[
    (saved_df["employee_id"] == employee_id) &
    (saved_df["observation_date"] == observation_date)
].iloc[0]

# Numeric feature columns
attendance_cols = [
    c for c in attendance_raw.select_dtypes(include="number").columns
    if c != "job_level"
]

payroll_cols = [
    c for c in payroll_raw.select_dtypes(include="number").columns
    if c != "job_level"
]

def verify_window(df, columns, prefix):

    history = df[
        (df["employee_id"] == employee_id) &
        (df["month"] >= window_start) &
        (df["month"] < observation_month)
    ].copy()

    # Consolidate duplicate employee-month records
    history = (
        history.groupby("month")[columns]
        .mean()
        .sort_index()
    )

    print("\nMonths used:", history.index.strftime("%Y-%m").tolist())

    results = []

    for col in columns:
        saved_col = f"{prefix}_{col}"

        if saved_col not in saved_df.columns:
            continue

        reconstructed = history[col].mean()
        original = saved_row[saved_col]

        results.append({
            "feature": saved_col,
            "saved_value": original,
            "reconstructed_value": reconstructed,
            "difference": abs(original - reconstructed),
            "match": np.isclose(
                original, reconstructed, atol=1e-6
            )
        })

    result_df = pd.DataFrame(results)
    display(result_df)

    print(
        "Exact matches:",
        result_df["match"].sum(),
        "/",
        len(result_df)
    )

print("\nATTENDANCE VERIFICATION")
verify_window(
    attendance_raw,
    attendance_cols,
    "attendance"
)

print("\nPAYROLL VERIFICATION")
verify_window(
    payroll_raw,
    payroll_cols,
    "payroll"
)

Expected history window:
2025-06-01 00:00:00 to 2025-09-01 00:00:00

ATTENDANCE VERIFICATION

Months used: ['2025-06', '2025-07', '2025-08']


,feature,saved_value,reconstructed_value,difference,match
0,attendance_scheduled_days,21.000000,21.000000,0.000000e+00,True
1,attendance_present_days,20.333333,20.333333,3.552714e-15,True
2,attendance_wfh_days,19.666667,19.666667,0.000000e+00,True
3,attendance_half_days_unpaid,0.333333,0.333333,0.000000e+00,True
4,attendance_paid_leave_days,0.333333,0.333333,0.000000e+00,True
5,attendance_unpaid_absence_days,0.000000,0.000000,0.000000e+00,True
6,attendance_longest_unpaid_streak_days,0.000000,0.000000,0.000000e+00,True
7,attendance_late_arrivals,1.000000,1.000000,0.000000e+00,True
8,attendance_avg_hours_worked,8.500000,8.500000,0.000000e+00,True
9,attendance_overtime_hours,0.000000,0.000000,0.000000e+00,True


Exact matches: 11 / 11

PAYROLL VERIFICATION

Months used: ['2025-06', '2025-07', '2025-08']


,feature,saved_value,reconstructed_value,difference,match
0,payroll_monthly_fixed_gross,144460.000000,144460.000000,0.000000e+00,True
1,payroll_payable_days,30.500000,30.500000,0.000000e+00,True
2,payroll_lop_days,0.166667,0.166667,5.551115e-17,True
3,payroll_overtime_hours_paid,0.000000,0.000000,0.000000e+00,True
4,payroll_overtime_pay,0.000000,0.000000,0.000000e+00,True
5,payroll_variable_pay,0.000000,0.000000,0.000000e+00,True
6,payroll_gross_pay,143657.443333,143657.443333,0.000000e+00,True
7,payroll_pf_employee,1800.000000,1800.000000,0.000000e+00,True
8,payroll_esi_employee,0.000000,0.000000,0.000000e+00,True
9,payroll_professional_tax,200.000000,200.000000,0.000000e+00,True


Exact matches: 14 / 14


In [41]:
print("SAVED INTEGRATION")
print("Shape:", saved_df.shape)
print("Columns:")
print(saved_df.columns.tolist())

print("\nATTENDANCE SOURCE")
print("Columns:")
print(attendance_raw.columns.tolist())

print("\nPAYROLL SOURCE")
print("Columns:")
print(payroll_raw.columns.tolist())

print("\nSAMPLE EMPLOYEE RECORDS")
print(
    saved_df[["employee_id", "observation_date"]]
    .head(10)
    .to_string(index=False)
)

SAVED INTEGRATION
Shape: (2389, 72)
Columns:
['row_id', 'employee_id', 'observation_date', 'department_id', 'role_id', 'job_level', 'work_location', 'work_mode', 'employment_type', 'highest_education', 'tenure_months', 'total_experience_years', 'previous_employers_count', 'commute_distance_km', 'annual_fixed_pay', 'comp_ratio', 'raises_received', 'last_raise_pct', 'months_since_last_raise', 'promotions_received', 'months_since_last_promotion', 'internal_transfers', 'last_performance_rating', 'last_engagement_score', 'manager_changes_12m', 'training_hours_12m', 'attendance_rate_3m', 'late_arrivals_3m', 'avg_overtime_hours_per_month_3m', 'avg_hours_worked_3m', 'wfh_share_3m', 'unpaid_leave_days_6m', 'paid_leave_days_6m', 'dept_attrition_rate_prior_12m', 'left_within_12m', 'voluntary_left_within_12m', 'source', 'exit_date', 'resignation_date', 'exit_reason', 'exit_interview_done', 'fnf_settlement_amount', 'attendance_scheduled_days', 'attendance_present_days', 'attendance_wfh_days', 'atte

In [43]:
import pandas as pd
import numpy as np

# Work on copies only
saved = saved_df.copy()
att = attendance_raw.copy()
pay = payroll_raw.copy()

saved["observation_date"] = pd.to_datetime(saved["observation_date"])
att["month"] = pd.to_datetime(att["month"]).dt.to_period("M")
pay["month"] = pd.to_datetime(pay["month"]).dt.to_period("M")

# Saved feature -> source monthly feature
att_map = {
    "attendance_scheduled_days": "scheduled_days",
    "attendance_present_days": "present_days",
    "attendance_wfh_days": "wfh_days",
    "attendance_half_days_unpaid": "half_days_unpaid",
    "attendance_paid_leave_days": "paid_leave_days",
    "attendance_unpaid_absence_days": "unpaid_absence_days",
    "attendance_longest_unpaid_streak_days": "longest_unpaid_streak_days",
    "attendance_late_arrivals": "late_arrivals",
    "attendance_avg_hours_worked": "avg_hours_worked",
    "attendance_overtime_hours": "overtime_hours",
    "attendance_attendance_rate": "attendance_rate"
}

pay_map = {
    "payroll_monthly_fixed_gross": "monthly_fixed_gross",
    "payroll_payable_days": "payable_days",
    "payroll_lop_days": "lop_days",
    "payroll_overtime_hours_paid": "overtime_hours_paid",
    "payroll_overtime_pay": "overtime_pay",
    "payroll_variable_pay": "variable_pay",
    "payroll_gross_pay": "gross_pay",
    "payroll_pf_employee": "pf_employee",
    "payroll_esi_employee": "esi_employee",
    "payroll_professional_tax": "professional_tax",
    "payroll_tds": "tds",
    "payroll_other_deductions": "other_deductions",
    "payroll_total_deductions": "total_deductions",
    "payroll_net_pay": "net_pay"
}

# One monthly value per employee and month.
# Duplicate rows are averaged within their month before the 3-month average.
att_monthly = (
    att.groupby(["employee_id", "month"])[list(att_map.values())]
    .mean()
)

pay_monthly = (
    pay.groupby(["employee_id", "month"])[list(pay_map.values())]
    .mean()
)

results = []

for _, row in saved.iterrows():

    emp = row["employee_id"]
    obs_month = row["observation_date"].to_period("M")

    # Three calendar months immediately before observation month
    history_months = [
        obs_month - 3,
        obs_month - 2,
        obs_month - 1
    ]

    for dataset, monthly_data, feature_map in [
        ("ATTENDANCE", att_monthly, att_map),
        ("PAYROLL", pay_monthly, pay_map)
    ]:

        available = [
            month for month in history_months
            if (emp, month) in monthly_data.index
        ]

        if len(available) != 3:
            continue

        history = monthly_data.loc[
            [(emp, month) for month in history_months]
        ]

        for saved_feature, source_feature in feature_map.items():

            reconstructed = history[source_feature].mean()
            saved_value = row[saved_feature]

            if pd.isna(reconstructed) or pd.isna(saved_value):
                continue

            difference = abs(saved_value - reconstructed)

            results.append({
                "employee_id": emp,
                "observation_date": row["observation_date"].date(),
                "dataset": dataset,
                "feature": saved_feature,
                "saved_value": saved_value,
                "reconstructed_value": reconstructed,
                "difference": difference,
                "match": np.isclose(
                    saved_value,
                    reconstructed,
                    rtol=1e-5,
                    atol=1e-6
                )
            })

verification = pd.DataFrame(results)

print("THREE-MONTH HISTORICAL INTEGRATION VERIFICATION")
print("=" * 65)

if verification.empty:
    print("No records had complete three-month histories.")
else:
    summary = (
        verification.groupby(["dataset", "feature"])["match"]
        .agg(["count", "sum"])
        .rename(columns={"count": "comparisons", "sum": "matches"})
    )
    summary["mismatches"] = (
        summary["comparisons"] - summary["matches"]
    )
    summary["match_percentage"] = (
        summary["matches"] / summary["comparisons"] * 100
    ).round(2)

    print("\nFEATURE-LEVEL RESULTS")
    print(summary.to_string())

    print("\nOVERALL RESULTS")
    print("Total comparisons:", len(verification))
    print("Matches:", int(verification["match"].sum()))
    print("Mismatches:", int((~verification["match"]).sum()))
    print(
        "Match percentage:",
        round(verification["match"].mean() * 100, 2), "%"
    )

    print("\nEMPLOYEE RECORDS VERIFIED")
    print(
        verification[
            ["employee_id", "observation_date"]
        ].drop_duplicates().shape[0]
    )

    print("\nFIRST 20 MISMATCHES")
    print(
        verification.loc[
            ~verification["match"]
        ].head(20).to_string(index=False)
    )

THREE-MONTH HISTORICAL INTEGRATION VERIFICATION

FEATURE-LEVEL RESULTS
                                                  comparisons  matches  mismatches  match_percentage
dataset    feature                                                                                  
ATTENDANCE attendance_attendance_rate                     368      368           0             100.0
           attendance_avg_hours_worked                    368      368           0             100.0
           attendance_half_days_unpaid                    368      368           0             100.0
           attendance_late_arrivals                       368      368           0             100.0
           attendance_longest_unpaid_streak_days          368      368           0             100.0
           attendance_overtime_hours                      368      368           0             100.0
           attendance_paid_leave_days                     368      368           0             100.0
           attendanc

In [44]:
print("SAVED INTEGRATION DUPLICATES")
print(
    "Duplicate employee/date rows:",
    saved.duplicated(
        ["employee_id", "observation_date"], keep=False
    ).sum()
)

print("\nATTENDANCE DUPLICATE EMPLOYEE-MONTHS")
print(
    att.groupby(["employee_id", "month"])
    .size().value_counts().sort_index().to_string()
)

print("\nPAYROLL DUPLICATE EMPLOYEE-MONTHS")
print(
    pay.groupby(["employee_id", "month"])
    .size().value_counts().sort_index().to_string()
)

SAVED INTEGRATION DUPLICATES
Duplicate employee/date rows: 12

ATTENDANCE DUPLICATE EMPLOYEE-MONTHS
1    7339

PAYROLL DUPLICATE EMPLOYEE-MONTHS
1    7325
2      15


In [45]:
# Inspect duplicate employee/date records in saved integration
saved_duplicates = saved[
    saved.duplicated(
        ["employee_id", "observation_date"],
        keep=False
    )
].sort_values(["employee_id", "observation_date"])

print("SAVED DUPLICATE RECORDS")
print(
    saved_duplicates[
        ["employee_id", "observation_date"] +
        list(att_map.keys()) +
        list(pay_map.keys())
    ].to_string(index=False)
)

# Inspect duplicate payroll employee-month records
payroll_duplicates = pay[
    pay.duplicated(
        ["employee_id", "month"],
        keep=False
    )
].sort_values(["employee_id", "month"])

print("\nPAYROLL DUPLICATE RECORDS")
print(
    payroll_duplicates.to_string(index=False)
)

SAVED DUPLICATE RECORDS
employee_id observation_date  attendance_scheduled_days  attendance_present_days  attendance_wfh_days  attendance_half_days_unpaid  attendance_paid_leave_days  attendance_unpaid_absence_days  attendance_longest_unpaid_streak_days  attendance_late_arrivals  attendance_avg_hours_worked  attendance_overtime_hours  attendance_attendance_rate  payroll_monthly_fixed_gross  payroll_payable_days  payroll_lop_days  payroll_overtime_hours_paid  payroll_overtime_pay  payroll_variable_pay  payroll_gross_pay  payroll_pf_employee  payroll_esi_employee  payroll_professional_tax  payroll_tds  payroll_other_deductions  payroll_total_deductions  payroll_net_pay
   EMP-0060       2025-09-30                       21.0                21.000000             7.333333                          0.0                    0.000000                        0.000000                               0.000000                  0.333333                     8.760000                   0.000000             

In [46]:
# Compare payroll duplicates after excluding their unique IDs
payroll_duplicate_groups = (
    pay.groupby(["employee_id", "month"])
    .filter(lambda x: len(x) > 1)
)

comparison_cols = [
    col for col in payroll_duplicate_groups.columns
    if col not in ["payroll_item_id"]
]

duplicate_comparison = (
    payroll_duplicate_groups
    .groupby(["employee_id", "month"])[comparison_cols]
    .nunique(dropna=False)
)

print("PAYROLL DUPLICATE CONTENT VERIFICATION")
print("All non-ID columns identical:",
      duplicate_comparison.eq(1).all().all())

print("\nMaximum distinct values per column:")
print(duplicate_comparison.max().to_string())

PAYROLL DUPLICATE CONTENT VERIFICATION
All non-ID columns identical: True

Maximum distinct values per column:
run_id                 1
employee_id            1
month                  1
department_id          1
role_id                1
job_level              1
overtime_eligible      1
monthly_fixed_gross    1
payable_days           1
lop_days               1
overtime_hours_paid    1
overtime_pay           1
variable_pay           1
variable_pay_type      1
gross_pay              1
pf_employee            1
esi_employee           1
professional_tax       1
tds                    1
other_deductions       1
total_deductions       1
net_pay                1
